# 3. Load claim evidence and run semantic hybrid search
Read prepared records, upload and verify the selected claim, then combine keyword search, a query vector and semantic reranking, following the [Azure vector-search sample](https://github.com/Azure/azure-search-vector-samples/blob/main/demo-python/code/basic-vector-workflow/azure-search-vector-python-sample.ipynb). This notebook does **not** call CU, regenerate document embeddings or create an index. The search cell generates only the question's embedding. Run notebooks 1 and 2 first.

Loading synchronizes only the selected claim. Deterministic IDs upsert on a retry; other claims are not deleted.

In [1]:
import os, json, math, hashlib
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from IPython.display import display
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizedQuery

ENV_FILE = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(ENV_FILE, override=True)
ROOT = ENV_FILE.parent
claim_folder = ROOT / os.getenv('CLAIM_FOLDER', r'data\claims\claim-001')
claim_id = claim_folder.name
claim_filter = "claim_id eq '" + claim_id.replace("'", "''") + "'"
output_folder = ROOT / os.getenv('CLAIMS_OUTPUT_ROOT', r'output\claims') / claim_id
processed_file = output_folder / 'processed-evidence-v1.json'
AZURE_SEARCH_SERVICE_ENDPOINT = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT', '').strip()
if not AZURE_SEARCH_SERVICE_ENDPOINT:
    raise ValueError(f'Set AZURE_SEARCH_SERVICE_ENDPOINT to the connected Search service endpoint in {ENV_FILE}; no upload or query was attempted.')
AZURE_SEARCH_ADMIN_CREDENTIAL = DefaultAzureCredential()
index_name = os.getenv('AZURE_SEARCH_INDEX_NAME', 'claims-evidence-v1').lower().replace('_', '-')
azure_openai_model_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072'))
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_seq_items', 3)

def ShowPreview(frame):
    preview = frame.head(5).copy()
    for column in preview.columns:
        if column.endswith('_vector'):
            preview[column] = preview[column].map(lambda vector: f'{len(vector)} dimensions')
        else:
            preview[column] = preview[column].map(lambda value: str(value)[:160])
    display(preview)

print('Index:', index_name, '| Claim:', claim_id)
print('Prepared file:', processed_file)

Index: claims-evidence-v1 | Claim: claim-001
Prepared file: c:\Users\jomedin\Documents\azure-ai-foundry-insurance\output\claims\ais-foundry-eval-dev-02\claim-001\processed-evidence-v1.json


## Step 1: Read and validate the claims-evidence records
Load every prepared row and its full contextual vector, but display only five rows, shortened text and vector dimensions. The versioned filename keeps older two-vector exports intact. No extraction or vectorization is repeated.

In [2]:
print('-' * 72)
print('Step 1: Reading processed data')
records = json.loads(processed_file.read_text(encoding='utf-8'))
index_columns = ['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'source_type', 'content_kind', 'filename', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region', 'analyzer_id', 'analyzer_api_version', 'content_vector']
if not isinstance(records, list) or not records or any(not isinstance(row, dict) or set(row) != set(index_columns) for row in records):
    raise ValueError('Prepared rows are empty or do not match the index columns. Rerun notebook 1.')
if any(row['claim_id'] != claim_id for row in records):
    raise ValueError('The prepared file contains a different claim.')
if len({row['id'] for row in records}) != len(records):
    raise ValueError('Prepared rows contain duplicate IDs.')
valid_kinds = {'document': {'document_text', 'figure_description'}, 'image': {'image_description', 'document_text', 'figure_description'}, 'email': {'email_text'}}
for row in records:
    vector = row['content_vector']
    if not isinstance(vector, list) or len(vector) != azure_openai_model_dimensions or any(type(value) not in (int, float) or not math.isfinite(value) for value in vector):
        raise ValueError('A content vector has incorrect dimensions or non-finite/non-numeric values.')
    if type(row['chunk_id']) is not int or row['chunk_id'] < 1:
        raise ValueError('Chunk numbers must be positive integers.')
    if row['content_kind'] not in valid_kinds.get(row['source_type'], set()):
        raise ValueError('Invalid source type/evidence kind combination.')
    if not isinstance(row['content'], str) or not row['content'].strip() or not row['source_path'] or not row['source_files'] or not row['analyzer_id']:
        raise ValueError('Evidence content and provenance must not be empty.')
    expected_id = hashlib.sha256(f"{claim_id}:{row['document_hash']}:{row['chunk_id']}".encode()).hexdigest()
    if row['id'] != expected_id:
        raise ValueError('A prepared evidence ID does not match its claim/document/chunk.')
df = pd.DataFrame(records, columns=index_columns)
ShowPreview(df)
print('Rows to upload:', len(df), '| Columns:', len(df.columns))

------------------------------------------------------------------------
Step 1: Reading processed data


,id,claim_id,document_hash,chunk_id,title,header,content,...,source_files,parent_emails,figure_id,source_region,analyzer_id,analyzer_api_version,content_vector
0,49d92297125cfe67adbf6889ba47af85921ba59d272fb651c1b6546ec961a6fd,claim-001,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,1,claim-form.pdf,,<!-- PageHeader: ASTER VALE | CLAIMS SERVICES --> \nFictional demonstration material | Not an insurance contract,...,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-documentSearch,2025-11-01,3072 dimensions
1,75d4be7842008c2e921dd2babb074696d168d7ad0fa6ccd901c43445eee55ba2,claim-001,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,2,First notification of loss,,# First notification of loss \nAV-26-004181 | Cedar Quay Warehousing LLC \nCLAIM REFERENCE\nAV-26-004181 \nPOLICY REFERENCE\nAV-CP-260101 \nINSURED / RE...,...,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-documentSearch,2025-11-01,3072 dimensions
2,dac5283c434b69f690efa44f2df19a3059b73ec45e18c1703164a2408e9f933c,claim-001,29c778e0261eb6326e50990e1440ef0ecc92ed63332ae845c6a6f93e6a5943ea,3,First notification of loss,Account supplied at notification,## Account supplied at notification \nThe site manager reported displaced roof sheets after overnight high winds. Water was\ndiscovered beside pallet stora...,...,"['claim-001/claim-form.pdf', 'claim-001/01-notification.msg::claim-form.pdf']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-documentSearch,2025-11-01,3072 dimensions
3,293119ce5a2c1a2755bb348da4a2aea2feaca39c59291ed4ac758866f85c21f2,claim-001,dced4e48ca24c1df7a8d417b8ce365a8561393d1d709778bfbc456d43633aee8,1,damage-01.jpg,Visual observation; not a causation or coverage finding,"The image shows the corner of a light-gray metal building with significant roof damage: several corrugated metal roof panels are bent upward and torn, expos...",...,"['claim-001/damage-01.jpg', 'claim-001/01-notification.msg::damage-01.jpg']",['8e65cd228a775af62c28ae1164216cbd53e71f8a2fc1f65aef3e7e5ad0d1b84a'],,,prebuilt-imageSearch,2025-11-01,3072 dimensions
4,d6202cc1ff3ad071b403f94c81b0924937c983798cc566a3948418f0fb34eef7,claim-001,b71b52334d5793e1588b1febcea4746f16af164e819e897903b9da00165f43a4,1,damage-02.jpg,Visual observation; not a causation or coverage finding,The image shows the interior of a metal storage shed or warehouse with corrugated metal walls and a translucent corrugated roof supported by a metal frame. ...,...,['claim-001/damage-02.jpg'],[],,,prebuilt-imageSearch,2025-11-01,3072 dimensions


Rows to upload: 33 | Columns: 18


## Step 2: Upload with batch summaries
Before uploading, check the target's text, vector and semantic configuration. Check every upload response and save full results. Preview failures instead of printing every successful row. A successful HTTP response alone does not establish that every row was accepted.

In [3]:
print('-' * 72)
print('Step 2: Uploading processed records')
with SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=AZURE_SEARCH_ADMIN_CREDENTIAL) as index_client:
    remote_index = index_client.get_index(index_name)
remote_fields = {field.name: field for field in remote_index.fields}
if set(remote_fields) != set(index_columns):
    raise ValueError('Target index has a different schema. Run notebook 2 with a new claims-evidence index name; nothing was uploaded or deleted.')
if not remote_fields['id'].key or not remote_fields['claim_id'].filterable or not remote_fields['content_vector'].searchable or not remote_fields['content_vector'].hidden or remote_fields['content_vector'].vector_search_dimensions != azure_openai_model_dimensions or remote_fields['content_vector'].vector_search_profile_name != 'claims-content-profile':
    raise ValueError('Target key, claim filter or vector configuration is incompatible. Nothing was uploaded or deleted.')
if any(not remote_fields[name].searchable or remote_fields[name].hidden or remote_fields[name].type != 'Edm.String' for name in ['title', 'header', 'content', 'filename']):
    raise ValueError('Hybrid search requires searchable, retrievable text fields. Nothing was uploaded or deleted.')
profiles = remote_index.vector_search.profiles if remote_index.vector_search else []
algorithms = remote_index.vector_search.algorithms if remote_index.vector_search else []
if not any(profile.name == 'claims-content-profile' and profile.algorithm_configuration_name == 'claims-hnsw' for profile in profiles or []) or not any(algorithm.name == 'claims-hnsw' and algorithm.kind == 'hnsw' and algorithm.parameters and algorithm.parameters.metric == 'cosine' for algorithm in algorithms or []):
    raise ValueError('Hybrid search requires the claims HNSW/cosine vector profile. Nothing was uploaded or deleted.')
semantic_configurations = remote_index.semantic_search.configurations if remote_index.semantic_search else []
semantic_configuration = next((configuration for configuration in semantic_configurations or [] if configuration.name == 'claims-semantic'), None)
prioritized = semantic_configuration.prioritized_fields if semantic_configuration else None
if not prioritized or not prioritized.title_field or prioritized.title_field.field_name != 'title' or [field.field_name for field in prioritized.content_fields or []] != ['content', 'header']:
    raise ValueError('Semantic hybrid search requires claims-semantic over title/content/header. Run notebook 2 to check the index. Nothing was uploaded or deleted.')
print('Index supports keyword + vector search and claims-semantic reranking.')
search_client = SearchClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, index_name=index_name, credential=AZURE_SEARCH_ADMIN_CREDENTIAL)
old_ids = {row['id'] for row in search_client.search(search_text='*', filter=claim_filter, select=['id'])}
upload_results = []
for start in range(0, len(records), 100):
    source_batch = records[start:start+100]
    batch = search_client.upload_documents(documents=source_batch)
    batch_results = [{'id': result.key, 'succeeded': result.succeeded, 'status_code': result.status_code, 'error': result.error_message} for result in batch]
    upload_results.extend(batch_results)
    print('Batch:', start // 100 + 1, '| Accepted:', sum(result.succeeded for result in batch), '/', len(source_batch))
    (output_folder / 'upload-results.json').write_text(json.dumps(upload_results, indent=2), encoding='utf-8')
    if len(batch) != len(source_batch) or any(not result.succeeded for result in batch):
        ShowPreview(pd.DataFrame([row for row in batch_results if not row['succeeded']]))
        raise RuntimeError('Some rows were not accepted. Inspect upload-results.json; stale rows were not removed.')
print('Uploaded or updated:', len(upload_results))

------------------------------------------------------------------------
Step 2: Uploading processed records
Index supports keyword + vector search and claims-semantic reranking.
Batch: 1 | Accepted: 33 / 33
Uploaded or updated: 33


## Step 3: Remove stale rows for this claim only
After a successful upload, remove old chunk IDs not present in the new prepared file. This does not affect another claim.

In [4]:
print('-' * 72)
print('Step 3: Synchronizing selected-claim rows')
new_ids = {row['id'] for row in records}
stale_ids = old_ids - new_ids
print('Stale rows for this claim:', len(stale_ids))
if stale_ids:
    deleted = search_client.delete_documents(documents=[{'id': key} for key in sorted(stale_ids)])
    delete_results = [{'id': result.key, 'succeeded': result.succeeded, 'error': result.error_message} for result in deleted]
    (output_folder / 'delete-results.json').write_text(json.dumps(delete_results, indent=2), encoding='utf-8')
    if len(deleted) != len(stale_ids) or any(not result.succeeded for result in deleted):
        ShowPreview(pd.DataFrame([row for row in delete_results if not row['succeeded']]))
        raise RuntimeError('Some stale rows could not be deleted; inspect delete-results.json.')
print('Other claims deleted: 0')

------------------------------------------------------------------------
Step 3: Synchronizing selected-claim rows
Stale rows for this claim: 0
Other claims deleted: 0


## Step 4: Verify what is visible in the index
Search visibility can lag an upload. Rerun this cell if needed, not notebook 1.

In [7]:
print('-' * 72)
print('Step 4: Verifying indexed records')
visible_rows = list(search_client.search(search_text='*', filter=claim_filter, select=['id', 'claim_id', 'filename', 'content_kind', 'chunk_id', 'figure_id']))
visible_ids = {row['id'] for row in visible_rows}
ShowPreview(pd.DataFrame(visible_rows))
print('Prepared rows:', len(new_ids), '| Visible rows:', len(visible_ids))
print('Missing IDs:', len(new_ids - visible_ids), '| First five:', sorted(new_ids - visible_ids)[:5])
print('Stale IDs:', len(visible_ids - new_ids), '| First five:', sorted(visible_ids - new_ids)[:5])
if visible_ids != new_ids:
    raise RuntimeError('The index is not synchronized yet. Rerun this verification cell after Search catches up.')
print('All prepared rows are visible for', claim_id)

------------------------------------------------------------------------
Step 4: Verifying indexed records


,figure_id,id,claim_id,content_kind,chunk_id,filename,@search.score,@search.reranker_score,@search.highlights,@search.captions
0,,49d92297125cfe67adbf6889ba47af85921ba59d272fb651c1b6546ec961a6fd,claim-001,document_text,1,claim-form.pdf,1.0,None,None,None
1,,293119ce5a2c1a2755bb348da4a2aea2feaca39c59291ed4ac758866f85c21f2,claim-001,image_description,1,damage-01.jpg,1.0,None,None,None
2,,631923ad26e6b4a6abd29709c808c2a63e316f2fe958cb4da3b308ede50ead99,claim-001,document_text,2,emergency-authorization.pdf,1.0,None,None,None
3,,dac5283c434b69f690efa44f2df19a3059b73ec45e18c1703164a2408e9f933c,claim-001,document_text,3,claim-form.pdf,1.0,None,None,None
4,,a3973e35c4a47207819c2126595d9de23446d9a28f6e7f761e583a26bb3e066f,claim-001,document_text,7,inspection.pdf,1.0,None,None,None


Prepared rows: 33 | Visible rows: 33
Missing IDs: 0 | First five: []
Stale IDs: 0 | First five: []
All prepared rows are visible for claim-001


## Step 5: Semantic hybrid search within this claim
Edit and run the question cell below, then run the search cell immediately after it. Setting the question makes no Azure calls. The search cell makes one query-embedding call using the same Foundry account, deployment and dimensions as notebook 1, authenticated with Entra ID. No document extraction, document embeddings or uploads are repeated.

The text query and `content_vector` search run together; Azure combines their ranks with RRF, then `claims-semantic` reranks the candidates. Request 50 vector candidates and return at most five results, prefiltered to the selected claim. A small claim can have fewer than 50 candidates. A claim filter scopes evidence; it is not authorization.

**Prerequisite:** semantic ranker must be enabled on the Search service in a supported region/tier. An index semantic configuration alone does not enable that service feature. Semantic errors are raised, not silently downgraded to ordinary hybrid search. No service settings are changed here.

Results show source, evidence kind, RRF score, semantic reranker score and an extractive caption. Scores are ranking signals, not probabilities or confidence. Captions are source excerpts, not generated claims decisions. Full selected evidence remains in `hybrid_results`; previews stay compact and never include vectors.

In [11]:
query = 'What are we still waiting on to move this claim forward?'

In [12]:
from openai import OpenAI
import tiktoken

print('-' * 72)
print('Step 5: Semantic hybrid search')
query = query.strip()
if not query or len(tiktoken.get_encoding('cl100k_base').encode(query, disallowed_special=())) > 8000:
    raise ValueError('Enter a nonempty question of at most 8000 tokens; do not truncate it silently.')
embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT') or os.getenv('EMBEDDING_MODEL_DEPLOYMENT')
if not embedding_deployment:
    raise ValueError('Set the same embedding deployment used in notebook 1 before searching.')
foundry_openai_endpoint = (os.getenv('AZURE_OPENAI_ENDPOINT') or os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT'].replace('.cognitiveservices.azure.com', '.openai.azure.com').replace('.services.ai.azure.com', '.openai.azure.com')).rstrip('/')
with DefaultAzureCredential() as query_credential:
    with OpenAI(base_url=f'{foundry_openai_endpoint}/openai/v1/', api_key=get_bearer_token_provider(query_credential, 'https://ai.azure.com/.default')) as query_client:
        embedding_response = query_client.embeddings.create(input=query, model=embedding_deployment, dimensions=azure_openai_model_dimensions)
    query_vector = embedding_response.data[0].embedding
    if len(query_vector) != azure_openai_model_dimensions or any(not math.isfinite(value) for value in query_vector):
        raise ValueError('Query embedding dimensions or values are incompatible with the index.')
    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=50, fields='content_vector')
    with SearchClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, index_name=index_name, credential=query_credential) as query_search_client:
        hybrid_results = list(query_search_client.search(
            search_text=query,
            search_fields=['title', 'header', 'content', 'filename'],
            vector_queries=[vector_query],
            filter=claim_filter,
            vector_filter_mode='preFilter',
            query_type='semantic',
            semantic_configuration_name='claims-semantic',
            semantic_error_mode='fail',
            query_caption='extractive',
            query_caption_highlight_enabled=False,
            select=['id', 'claim_id', 'document_hash', 'chunk_id', 'title', 'header', 'content', 'filename', 'content_kind', 'source_path', 'source_files', 'parent_emails', 'figure_id', 'source_region'],
            top=5,
        ))
if not hybrid_results:
    raise RuntimeError('No evidence matched this claim. Check the claim filter and Step 4 verification.')
if any(row['claim_id'] != claim_id for row in hybrid_results):
    raise RuntimeError('A result escaped the selected-claim filter.')
if any(row.get('@search.reranker_score') is None for row in hybrid_results):
    raise RuntimeError('Semantic reranking was not returned; check service enablement and claims-semantic.')
search_df = pd.DataFrame([{'rank': rank, 'filename': row['filename'], 'content_kind': row['content_kind'], 'rrf_score': row['@search.score'], 'semantic_score': row['@search.reranker_score'], 'caption': '\n'.join(caption.text for caption in row.get('@search.captions') or []), 'content': row['content']} for rank, row in enumerate(hybrid_results, start=1)])
ShowPreview(search_df)
print('Claim:', claim_id, '| Results:', len(hybrid_results), '| Vector candidates requested: 50 | Semantic: claims-semantic')

------------------------------------------------------------------------
Step 5: Semantic hybrid search


,rank,filename,content_kind,rrf_score,semantic_score,caption,content
0,1,03-adjuster-follow-up.msg,email_text,0.02916666865348816,1.7639235258102417,# RE: AV-26-004181 - outstanding evidence subject: RE: AV-26-004181 - outstanding evidence sender: Alex Rowan to: claims@example.invalid date: 2026-09-14...,# RE: AV-26-004181 - outstanding evidence \nsubject: RE: AV-26-004181 - outstanding evidence\nsender: Alex Rowan\nto: claims@example.invalid <None>\ndate: ...
1,2,02-inspection-and-attachments.msg,email_text,0.032522473484277725,1.6951565742492676,"# AV-26-004181 - illustrated report, authority and access note subject: AV-26-004181 - illustrated report, authority and access note sender: Meridian Fiel...","# AV-26-004181 - illustrated report, authority and access note \nsubject: AV-26-004181 - illustrated report, authority and access note\nsender: Meridian Fi..."
2,3,01-notification.msg,email_text,0.03100961446762085,1.6887761354446411,# AV-26-004181 - notification and site photograph subject: AV-26-004181 - notification and site photograph sender: Morgan Hale to: claims@example.invalid ...,# AV-26-004181 - notification and site photograph \nsubject: AV-26-004181 - notification and site photograph\nsender: Morgan Hale\nto: claims@example.inval...
3,4,claim-form.pdf,document_text,0.028624001890420914,1.6185911893844604,# First notification of loss AV-26-004181 | Cedar Quay Warehousing LLC CLAIM REFERENCE AV-26-004181 POLICY REFERENCE AV-CP-260101 INSURED / REPORTIN...,# First notification of loss \nAV-26-004181 | Cedar Quay Warehousing LLC \nCLAIM REFERENCE\nAV-26-004181 \nPOLICY REFERENCE\nAV-CP-260101 \nINSURED / RE...
4,5,policy.pdf,document_text,0.03028232976794243,1.5845621824264526,# Conditions and claim notification AV-26-004181 | Cedar Quay Warehousing LLC.,# Conditions and claim notification \nAV-26-004181 | Cedar Quay Warehousing LLC


Claim: claim-001 | Results: 5 | Vector candidates requested: 50 | Semantic: claims-semantic


## Close loading clients
The search cell manages its own clients, so you can still edit and run the question cell, then rerun the search cell after this cleanup without repeating uploads.

In [8]:
search_client.close()
AZURE_SEARCH_ADMIN_CREDENTIAL.close()